In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = './'

print("Đang load dữ liệu...")
orders      = pd.read_csv(DATA_DIR + "orders.csv",      parse_dates=["order_date"])
products    = pd.read_csv(DATA_DIR + "products.csv")
returns     = pd.read_csv(DATA_DIR + "returns.csv",     parse_dates=["return_date"])
web_traffic = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
order_items = pd.read_csv(DATA_DIR + "order_items.csv", low_memory=False)
customers  = pd.read_csv(DATA_DIR + 'customers.csv')
geography  = pd.read_csv(DATA_DIR + 'geography.csv')
inventory  = pd.read_csv(DATA_DIR + 'inventory.csv')
payments  = pd.read_csv(DATA_DIR + 'payments.csv')
reviews    = pd.read_csv(DATA_DIR + 'reviews.csv')
shipments  = pd.read_csv(DATA_DIR + 'shipments.csv')
promotions = pd.read_csv(DATA_DIR + 'promotions.csv')
sales  = pd.read_csv(DATA_DIR + 'sales.csv')
sample_submission  = pd.read_csv(DATA_DIR + 'sample_submission.csv')
print("Load xong!\n")


Đang load dữ liệu...
Load xong!



In [13]:
# Q1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

print("Q1: Trung vị inter-order gap của khách mua >1 lần")

multi_order_customers = (
    orders.groupby("customer_id")
    .filter(lambda x: len(x) > 1)
)
multi_order_customers = multi_order_customers.sort_values(
    ["customer_id", "order_date"]
)
gaps = (
    multi_order_customers
    .groupby("customer_id")["order_date"]
    .diff()          # khoảng cách giữa hai đơn liên tiếp
    .dt.days         # chuyển sang số ngày
    .dropna()        # bỏ dòng NaN (dòng đầu tiên của mỗi khách)
)

median_gap = gaps.median()
 
print(f"  Số khách hàng có >1 đơn  : {multi_order_customers['customer_id'].nunique():,}")
print(f"  Tổng số khoảng cách       : {len(gaps):,}")
print(f"  Trung vị inter-order gap  : {median_gap:.1f} ngày")

print(">>> ĐÁP ÁN: C) 144 ngày")
print()
 

Q1: Trung vị inter-order gap của khách mua >1 lần
  Số khách hàng có >1 đơn  : 67,888
  Tổng số khoảng cách       : 556,699
  Trung vị inter-order gap  : 144.0 ngày
>>> ĐÁP ÁN: C) 144 ngày



In [3]:
# Q2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?
# Tính từ products.csv

print("Q2: Segment có tỷ suất lợi nhuận gộp trung bình cao nhất")

products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]
margin_by_segment = (
    products
    .groupby("segment")["gross_margin"]
    .mean()
    .sort_values(ascending=False)
    .rename("Avg Gross Margin")
)

print("Gross margin trung bình theo Segment:")
for seg, val in margin_by_segment.items():
    marker = " => CAO NHẤT" if seg == margin_by_segment.idxmax() else ""
    print(f"    {seg:<15} {val:.4f}  ({val*100:.2f}%){marker}")
print()
print(">>> ĐÁP ÁN: D) Standard")
print()

Q2: Segment có tỷ suất lợi nhuận gộp trung bình cao nhất
Gross margin trung bình theo Segment:
    Standard        0.3134  (31.34%) => CAO NHẤT
    Premium         0.2854  (28.54%)
    All-weather     0.2842  (28.42%)
    Activewear      0.2656  (26.56%)
    Performance     0.2636  (26.36%)
    Balanced        0.2580  (25.80%)
    Trendy          0.2408  (24.08%)
    Everyday        0.2363  (23.63%)

>>> ĐÁP ÁN: D) Standard



In [2]:
# Q3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
# Tính từ returns.csv + products.csv (join theo product_id)

print("Q3: Lý do trả hàng nhiều nhất trong danh mục Streetwear")


streetwear_ids = products.loc[
    products["category"] == "Streetwear", "product_id"
]
streetwear_returns = returns[returns["product_id"].isin(streetwear_ids)]

reason_counts = streetwear_returns["return_reason"].value_counts()

print()
print("  Lý do trả hàng (sắp xếp từ nhiều nhất đến ít nhất):")
for reason, count in reason_counts.items():
    pct    = count / len(streetwear_returns) * 100
    marker = " <= PHỔ BIẾN NHẤT" if reason == reason_counts.idxmax() else ""
    print(f"    {reason:<20} {count:>6,}  ({pct:.1f}%){marker}")
print()
print(">>> ĐÁP ÁN Q3: B) wrong_size")
print()

Q3: Lý do trả hàng nhiều nhất trong danh mục Streetwear

  Lý do trả hàng (sắp xếp từ nhiều nhất đến ít nhất):
    wrong_size            7,626  (35.0%) <= PHỔ BIẾN NHẤT
    defective             4,330  (19.9%)
    not_as_described      3,854  (17.7%)
    changed_mind          3,830  (17.6%)
    late_delivery         2,159  (9.9%)

>>> ĐÁP ÁN Q3: B) wrong_size



In [5]:
# Q4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
# bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
# Tính từ web_traffic.csv
print("Q4: Traffic source có bounce_rate trung bình thấp nhất")

avg_bounce = (
    web_traffic
    .groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
    .rename("Avg Bounce Rate")
)
print("  Bounce rate trung bình theo nguồn:")
for src, val in avg_bounce.items():
    marker = " => THẤP NHẤT" if src == avg_bounce.idxmin() else ""
    print(f"    {src:<20} {val:.6f}{marker}")
print()
print(">>> ĐÁP ÁN: C) email_campaign")
print()

Q4: Traffic source có bounce_rate trung bình thấp nhất
  Bounce rate trung bình theo nguồn:
    email_campaign       0.004458 => THẤP NHẤT
    social_media         0.004476
    paid_search          0.004478
    referral             0.004499
    organic_search       0.004504
    direct               0.004511

>>> ĐÁP ÁN: C) email_campaign



In [6]:
# Q5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?
# Tính từ order_items.csv
print("Q5: % dòng order_items có promo_id không null")

total_rows  = len(order_items)
promo_rows  = order_items["promo_id"].notna().sum()
pct_promo   = promo_rows / total_rows * 100

print(f"  Tổng số dòng trong order_items : {total_rows:>10,}")
print(f"  Dòng có promo_id (không null)  : {promo_rows:>10,}")
print(f"  Dòng không có promo_id (null)  : {total_rows - promo_rows:>10,}")
print(f"  Tỷ lệ có khuyến mãi           : {pct_promo:>10.2f}%")
print()
print(">>> ĐÁP ÁN: C) 39%")  #chính xác: {:.2f}%)".format(pct_promo) - 38.66%
print()

Q5: % dòng order_items có promo_id không null
  Tổng số dòng trong order_items :    714,669
  Dòng có promo_id (không null)  :    276,316
  Dòng không có promo_id (null)  :    438,353
  Tỷ lệ có khuyến mãi           :      38.66%

>>> ĐÁP ÁN: C) 39%



In [7]:
# Q6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)
merge_df = pd.merge(customers.dropna(subset=['age_group']), orders, on='customer_id', how = 'inner')

result_6 = (
    merge_df
    .groupby('age_group')
    .agg(total_orders=('order_id', 'count'),
         unique_customers=('customer_id', 'nunique')
    )
    .reset_index()
)

result_6['avg_order_per_customer'] = result_6['total_orders'] / result_6['unique_customers']
result_6 = result_6[['age_group', 'avg_order_per_customer']].sort_values('avg_order_per_customer')
print(result_6)
# Đáp án A

  age_group  avg_order_per_customer
0     18-24                7.068577
1     25-34                7.112230
2     35-44                7.206159
3     45-54                7.220264
4       55+                7.268731


In [8]:
# Q7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?
merge_df = (order_items.merge(orders, on='order_id', how = 'inner').merge(geography, on='zip', how='inner'))

merge_df['revenue'] = (merge_df['quantity'] * merge_df['unit_price']) - merge_df['discount_amount']

result_7 = (
     merge_df
    .groupby('region')['revenue']
    .sum()
    .round(2)
    .reset_index(name='total_revenue')
    .sort_values('total_revenue', ascending=False)
)
print(result_7)
# Đáp án C

    region  total_revenue
1     East   7.291151e+09
0  Central   4.719491e+09
2     West   3.670227e+09


In [9]:

# Q8: Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?
result_8 = (
    orders[orders['order_status'] == 'cancelled']
    .groupby('payment_method')
    .size()
    .reset_index(name='buy')
    .sort_values('buy', ascending=False)
)
print(result_8)
# Đáp án A

  payment_method    buy
3    credit_card  28452
2            cod  15468
4         paypal   7817
0      apple_pay   5190
1  bank_transfer   2535


In [10]:
# Q9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?
merge = (order_items.merge(products, on='product_id',how='inner')
         .merge(returns, on=['order_id','product_id'],how = 'left', indicator=True))
merge['is_returned'] = (merge['_merge'] == 'both').astype(int)

result_9 = (
    merge
    .groupby('size')
    .agg(total_returns=('is_returned', 'sum'),total_items=('order_id', 'count'))
    .reset_index()
)

result_9['return_rate'] = result_9['total_returns'] / result_9['total_items']
result_9 = result_9[['size', 'return_rate']].sort_values('return_rate', ascending=False)
print(result_9)
#Đáp án A



  size  return_rate
2    S     0.056515
0    L     0.056250
1    M     0.055682
3   XL     0.055200


In [11]:
# Q10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
result_10 = (
    payments
    .groupby('installments')['payment_value']
    .mean()
    .reset_index(name = 'avg_payment')
    .sort_values('avg_payment', ascending=False)
)
print(result_10)
# Đáp án C

   installments   avg_payment
3             6  24446.654403
2             3  24399.635486
4            12  24245.772694
0             1  24113.274166
1             2    708.473729
